# SEACO-6076: genesis range funnel for one incremental run

Takes **one run** of `Incremental_New_Model_E2E_Prod` and follows every genesis (OTL) range through the pipeline,
using the per-run copies in `backup_database_incremental` (written by `IncrementalBackupRunner`, tagged with `runId` + `backupDate`).

| Step | Code | Backup table | What can drop a genesis range |
|---|---|---|---|
| 0 Genesis input | `ImpactedFeatureFetcher` / `PopulateDataInOtlRanges` | `otl_feature_address_range` | – |
| 1 Merger input filter | `Merger.merge` | – | `violations` not empty, no house numbers, country not in `country_scope`, unnamed road |
| 2 Merge (ARC vs OTL) | `ArcOtlRangeMerger` | `feature_address_range_merged` (`preference` = ARC / OTL) | **ARC has a range on the same road side → ARC wins** |
| 3 QA | `QualityRulesExecutor` | `qa_feature_address_range_merged` | `violationType = 'Error'` (dropped later) |
| 4 Overlap correction | `OverlapCorrectorServiceIncremental` | `overlap_feature_address_range_merged` | duplicate / overlapping range removed |
| 5 Adjust | `AddressRangeAdjusterService` | `adjusted_feature_address_range` | – (only changes LR) |
| 6 Delta vs AR snapshot | `DeltaDetector` | `delta_detected_range` | `Error` ranges filtered; `operation = none` → identical to production, nothing published |
| 7 Gold metadata | `IncrementalMetadataPopulation` | `processed_data_incremental.gold_metadata` | cross-check only |

**Reaches Orbis** = the side survives to step 6 with `preference = 'OTL'` and the delta has `add` / `update` (published),
or `none` (genesis range already in production, unchanged).

Outputs (displayed only): a funnel table, a status summary, and two range lists:
`to_orbis` (ranges that reach Orbis) and `discarded` (ranges dropped, with the step and reason).

Read-only on every table. Nothing is written to files.

In [0]:
# Inputs for the notebook: where the backup and gold tables live and which run to analyse.
# Leave run_id blank to use the latest run found in the backup.
dbutils.widgets.text("backup_db", "backup_database_incremental", "Backup schema (catalog.schema if UC)")
dbutils.widgets.text("gold_db", "processed_data_incremental", "Gold schema (gold_metadata)")
dbutils.widgets.text("country_scope_table", "", "country_scope table (optional, e.g. <input_db>.country_scope)")
dbutils.widgets.text("run_id", "", "runId (blank = latest run in backup)")

BACKUP_DB = dbutils.widgets.get("backup_db").strip()
GOLD_DB = dbutils.widgets.get("gold_db").strip()
COUNTRY_SCOPE_TABLE = dbutils.widgets.get("country_scope_table").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()
print(BACKUP_DB, GOLD_DB, COUNTRY_SCOPE_TABLE or "<no country scope check>", RUN_ID or "<latest>")

## 1. Check the backup tables exist and look at their schema

In [0]:
# Names the 7 tables the analysis reads and checks they exist.
# Also prints the types of the key columns (roadId / id, side, operation, preference) so the joins can be trusted.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def norm_side(c):
    # works whether side is stored as 'left' or as a struct / enum string like '{left}'
    return F.lower(F.regexp_extract(F.col(c).cast("string"), "(?i)(left|right|both)", 1))

T_OTL     = f"{BACKUP_DB}.otl_feature_address_range"
T_MERGED  = f"{BACKUP_DB}.feature_address_range_merged"
T_QA      = f"{BACKUP_DB}.qa_feature_address_range_merged"
T_OVERLAP = f"{BACKUP_DB}.overlap_feature_address_range_merged"
T_ADJ     = f"{BACKUP_DB}.adjusted_feature_address_range"
T_DELTA   = f"{BACKUP_DB}.delta_detected_range"
T_META    = f"{GOLD_DB}.gold_metadata"

for t in [T_OTL, T_MERGED, T_QA, T_OVERLAP, T_ADJ, T_DELTA, T_META]:
    try:
        cols = spark.table(t).columns
        print(f"OK   {t}: {len(cols)} cols")
    except Exception as e:
        print(f"MISSING {t}: {str(e).splitlines()[0][:150]}")

# side is an enum in the Scala model; check how it is stored (string vs struct) before trusting the joins
for t in [T_OTL, T_MERGED, T_DELTA]:
    print(t, [f"{f.name}:{f.dataType.simpleString()[:60]}" for f in spark.table(t).schema if f.name in ("roadId", "id", "side", "operation", "preference", "runId", "backupDate")])

## 2. Pick the run

In [0]:
# Lists the most recent runs in the backup (genesis range count per run) and picks the run to analyse.
# runId = Databricks parentRunId of the job (numeric string), backupDate = dd-MM-yyyy
runs = (spark.table(T_OTL)
        .groupBy("runId", "backupDate")
        .agg(F.count("*").alias("otl_ranges"))
        .withColumn("backup_dt", F.to_date("backupDate", "dd-MM-yyyy"))
        .orderBy(F.col("backup_dt").desc(), F.col("runId").cast("bigint").desc()))
display(runs.limit(15))

if not RUN_ID:
    RUN_ID = runs.first()["runId"]
print("Using runId =", RUN_ID)

## 2b. Stats for every run in the backup

One row per run. Range counts come from `feature_address_range_merged`, the merge output.
Delta counts come from `delta_detected_range`, which has one row per tag, so they are counted as distinct **road sides**.

| Column | Meaning |
|---|---|
| `total_count_of_ranges` | ranges after the ARC/OTL merge |
| `arc_ranges` / `otl_ranges` | of those, where ARC won the side / where genesis won the side |
| `otl_delta_add` | genesis-won sides that are new in Orbis |
| `otl_delta_update` | genesis-won sides whose range changed |
| `otl_delta_delete` | genesis-won sides where a tag was removed (non-scrapped delete) |
| `delta_delete_scrapped_any_source` | sides whose range was removed entirely; the backup can't tell if it was ARC or genesis |
| `otl_ranges_produced` | raw genesis output for the run, before any filter |

In [0]:
# Builds the per-run stats table above for every runId in the backup.
# Delta rows are attributed to genesis by joining on the (runId, roadId, side) that OTL won in the merge.
RK = ["runId", "roadId", "side_n"]

merged_all = spark.table(T_MERGED).withColumn("side_n", norm_side("side"))
range_stats = merged_all.groupBy("runId").agg(
    F.first("backupDate").alias("date"),
    F.count("*").alias("total_count_of_ranges"),
    F.sum((F.col("preference") == "ARC").cast("int")).alias("arc_ranges"),
    F.sum((F.col("preference") == "OTL").cast("int")).alias("otl_ranges"))

otl_won_all = merged_all.filter(F.col("preference") == "OTL").select(*RK).distinct()

delta_all = (spark.table(T_DELTA).withColumnRenamed("id", "roadId")
             .withColumn("side_n", norm_side("side"))
             .withColumn("op", F.lower(F.col("operation").cast("string")))
             .withColumn("scrapped", F.coalesce(F.col("scrapped"), F.lit(False))))

side_key = F.struct("roadId", "side_n")
otl_delta = (delta_all.join(otl_won_all, RK).groupBy("runId").agg(
    F.countDistinct(F.when(F.col("op") == "add", side_key)).alias("otl_delta_add"),
    F.countDistinct(F.when(F.col("op") == "update", side_key)).alias("otl_delta_update"),
    F.countDistinct(F.when((F.col("op") == "delete") & ~F.col("scrapped"), side_key)).alias("otl_delta_delete")))

scrapped = (delta_all.filter((F.col("op") == "delete") & F.col("scrapped")).groupBy("runId")
            .agg(F.countDistinct(side_key).alias("delta_delete_scrapped_any_source")))

produced = spark.table(T_OTL).groupBy("runId").agg(F.count("*").alias("otl_ranges_produced"))

run_stats = (range_stats.join(otl_delta, "runId", "left")
             .join(scrapped, "runId", "left")
             .join(produced, "runId", "left")
             .fillna(0)
             .withColumnRenamed("runId", "run_id")
             .withColumn("_d", F.to_date("date", "dd-MM-yyyy"))
             .orderBy(F.col("_d").desc(), F.col("run_id").cast("bigint").desc())
             .drop("_d")
             .select("run_id", "date", "total_count_of_ranges", "arc_ranges", "otl_ranges",
                     "otl_delta_add", "otl_delta_update", "otl_delta_delete",
                     "delta_delete_scrapped_any_source", "otl_ranges_produced"))
display(run_stats)

## 3. Load the run's data, one row per range

In [0]:
# Loads the chosen run from every backup table, one row per range, and applies Merger's input filters to the genesis ranges.
# Later tables are kept only where OTL won the merge (preference = 'OTL'); the delta's `id` is renamed to `roadId`.
def for_run(t):
    return spark.table(t).filter(F.col("runId") == RUN_ID)

named_road = F.exists(F.col("tags"), lambda t: (t["tagKey"]["key"] == "name") & F.coalesce(F.length(t["value"]) > 0, F.lit(False)))
hn_count = F.coalesce(F.size("lrHouseNumbers"), F.lit(0))
hn_count = F.when(hn_count < 0, 0).otherwise(hn_count)  # size(null) = -1

otl = (for_run(T_OTL)
       .withColumn("side_n", norm_side("side"))
       .withColumn("hn_count", hn_count)
       .withColumn("is_named", F.coalesce(named_road, F.lit(False))))

if COUNTRY_SCOPE_TABLE:
    scope = spark.table(COUNTRY_SCOPE_TABLE).select(F.col("Country_ISO").alias("scope_iso")).distinct()
    otl = otl.join(scope, otl.countryIso == scope.scope_iso, "left").withColumn("in_scope", F.col("scope_iso").isNotNull()).drop("scope_iso")
else:
    otl = otl.withColumn("in_scope", F.lit(True))

# Merger.merge input filter, in the order the code applies it
otl = otl.withColumn(
    "input_drop_reason",
    F.when(F.trim(F.coalesce(F.col("violations"), F.lit(""))) != "", "GENESIS_VIOLATION")
     .when(F.col("hn_count") == 0, "NO_HOUSE_NUMBERS")
     .when(~F.col("in_scope"), "COUNTRY_NOT_IN_SCOPE")
     .when(~F.col("is_named"), "UNNAMED_ROAD"))
otl = otl.cache()

def side_keys(df):
    return df.withColumn("side_n", norm_side("side"))

merged  = side_keys(for_run(T_MERGED))
qa      = side_keys(for_run(T_QA)).filter(F.col("preference") == "OTL")
overlap = side_keys(for_run(T_OVERLAP)).filter(F.col("preference") == "OTL")
adj     = side_keys(for_run(T_ADJ)).filter(F.col("preference") == "OTL")
# DeltaDetectedRange names the road key `id`, the range tables name it `roadId`
delta   = (side_keys(for_run(T_DELTA)).withColumnRenamed("id", "roadId")
           .withColumn("op", F.lower(F.col("operation").cast("string"))))

print("otl ranges in run:", otl.count())

## 4. Status of every genesis road side at each step

In [0]:
# For every genesis road side, flags whether it survived each step (filter, merge, QA, overlap, adjust, delta).
# side_status = the first step it failed, or TO_ORBIS_ADD / UPDATE / UNCHANGED if it reached the end.
K = ["roadId", "side_n"]

def flag(df, name, cond=F.lit(True)):
    return df.filter(cond).select(*K).distinct().withColumn(name, F.lit(True))

not_error = F.coalesce(F.col("violationType"), F.lit("")) != "Error"
has_hn = F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0

s_eligible = flag(otl, "s1_eligible", F.col("input_drop_reason").isNull())
s_otl_won  = flag(merged, "s2_otl_won", F.col("preference") == "OTL")
s_arc_won  = flag(merged, "s2_arc_won", F.col("preference") == "ARC")
s_qa       = flag(qa, "s3_in_qa")
s_qa_ok    = flag(qa, "s3_qa_ok", not_error)
s_overlap  = flag(overlap, "s4_in_overlap")
s_adj_ok   = flag(adj, "s5_adj_ok", not_error & has_hn)

# delta: one row per tag; roll up to the side. Non-scrapped delete = a tag key removed from a range that is still there.
s_delta = (delta.groupBy(*K).agg(
              F.max(F.when(F.col("op") == "add", 1).otherwise(0)).alias("d_add"),
              F.max(F.when(F.col("op") == "update", 1).otherwise(0)).alias("d_update"),
              F.max(F.when((F.col("op") == "delete") & ~F.coalesce(F.col("scrapped"), F.lit(False)), 1).otherwise(0)).alias("d_del_tag"),
              F.max(F.when(F.col("op") == "none", 1).otherwise(0)).alias("d_none"))
           .withColumn("delta_op",
               F.when(F.col("d_add") == 1, "add")
                .when((F.col("d_update") == 1) | (F.col("d_del_tag") == 1), "update")
                .when(F.col("d_none") == 1, "none"))
           .select(*K, "delta_op"))

sides = otl.select(*K).distinct()
for f in [s_eligible, s_otl_won, s_arc_won, s_qa, s_qa_ok, s_overlap, s_adj_ok]:
    sides = sides.join(f, K, "left")
FLAGS = ["s1_eligible", "s2_otl_won", "s2_arc_won", "s3_in_qa", "s3_qa_ok", "s4_in_overlap", "s5_adj_ok"]
sides = sides.join(s_delta, K, "left").fillna(False, subset=FLAGS)

sides = sides.withColumn("side_status",
    F.when(~F.col("s1_eligible"), "DISCARDED_INPUT_FILTER")
     .when(F.col("s2_arc_won"), "DISCARDED_ARC_WINS")
     .when(~F.col("s2_otl_won"), "DISCARDED_NOT_IN_MERGE")
     .when(~F.col("s3_in_qa"), "DISCARDED_NOT_IN_QA")
     .when(~F.col("s3_qa_ok"), "DISCARDED_QA_ERROR")
     .when(~F.col("s4_in_overlap"), "DISCARDED_OVERLAP_CORRECTION")
     .when(~F.col("s5_adj_ok"), "DISCARDED_AFTER_ADJUST")
     .when(F.col("delta_op") == "add", "TO_ORBIS_ADD")
     .when(F.col("delta_op") == "update", "TO_ORBIS_UPDATE")
     .when(F.col("delta_op") == "none", "TO_ORBIS_UNCHANGED")
     .otherwise("DISCARDED_NO_DELTA")).cache()

print("genesis road sides in run:", sides.count())

## 5. Funnel: genesis road sides left after each step

In [0]:
# Funnel: how many genesis road sides are left after each step, then the count and % per final status.
funnel = sides.agg(
    F.count("*").alias("0_genesis_sides"),
    F.sum(F.col("s1_eligible").cast("int")).alias("1_pass_merger_filter"),
    F.sum(F.col("s2_otl_won").cast("int")).alias("2_otl_won_merge"),
    F.sum((F.col("s2_otl_won") & F.col("s3_qa_ok")).cast("int")).alias("3_pass_qa"),
    F.sum((F.col("s2_otl_won") & F.col("s3_qa_ok") & F.col("s4_in_overlap")).cast("int")).alias("4_pass_overlap"),
    F.sum((F.col("s2_otl_won") & F.col("s3_qa_ok") & F.col("s4_in_overlap") & F.col("s5_adj_ok")).cast("int")).alias("5_pass_adjust"),
    F.sum(F.col("side_status").startswith("TO_ORBIS").cast("int")).alias("6_reach_orbis"),
    F.sum(F.col("side_status").isin("TO_ORBIS_ADD", "TO_ORBIS_UPDATE").cast("int")).alias("6a_published_add_or_update"),
)
display(funnel)

display(sides.groupBy("side_status").count()
        .withColumn("pct", F.round(100 * F.col("count") / F.sum("count").over(Window.partitionBy()), 2))
        .orderBy(F.desc("count")))

## 6. Range-level view (ranges, roads, house numbers) with the reason per range

In [0]:
# Puts the side status back on each genesis range (input-filter drops keep their own reason) and adds QA violation text.
# Summarises ranges, road sides, roads and house numbers per status.
ranges = (otl.join(sides.select(*K, "side_status", "delta_op"), K, "left")
          .withColumn("status",
              F.when(F.col("input_drop_reason").isNotNull(), F.concat(F.lit("DISCARDED_"), F.col("input_drop_reason")))
               .otherwise(F.col("side_status")))
          .withColumn("reaches_orbis", F.col("status").startswith("TO_ORBIS")))

# QA violation text, to explain DISCARDED_QA_ERROR
qa_viol = (qa.filter(~not_error).groupBy(*K)
           .agg(F.concat_ws(" | ", F.collect_set("violations")).alias("qa_violations")))
ranges = ranges.join(qa_viol, K, "left").cache()

summary = (ranges.groupBy("reaches_orbis", "status")
           .agg(F.count("*").alias("ranges"),
                F.countDistinct(*K).alias("road_sides"),
                F.countDistinct("roadId").alias("roads"),
                F.sum("hn_count").alias("house_numbers"))
           .orderBy(F.desc("reaches_orbis"), F.desc("ranges")))
display(summary)

display(ranges.groupBy("reaches_orbis")
        .agg(F.count("*").alias("ranges"), F.countDistinct(*K).alias("road_sides"),
             F.countDistinct("roadId").alias("roads"), F.sum("hn_count").alias("house_numbers")))

### Per country

In [0]:
# Per country: genesis sides, how many reached Orbis, how many lost to ARC, and the % that reached Orbis.
display(ranges.groupBy("countryIso")
        .agg(F.countDistinct(*K).alias("genesis_sides"),
             F.countDistinct(F.when(F.col("reaches_orbis"), F.struct(*K))).alias("sides_to_orbis"),
             F.countDistinct(F.when(F.col("status") == "DISCARDED_ARC_WINS", F.struct(*K))).alias("sides_arc_wins"))
        .withColumn("pct_to_orbis", F.round(100 * F.col("sides_to_orbis") / F.col("genesis_sides"), 2))
        .orderBy(F.desc("genesis_sides")))

## 7. Cross-check against gold_metadata (sides written as GENESIS by this run)

In [0]:
# Sanity check: sides this run published as genesis (add/update) vs gold_metadata rows written by this run as GENESIS.
meta_run = (spark.table(T_META)
            .filter((F.col("lastUpdatedOnRunId") == RUN_ID) & (F.col("sourceType") == "GENESIS"))
            .withColumn("side_n", norm_side("side")).select(*K).distinct())
pipeline_pub = sides.filter(F.col("side_status").isin("TO_ORBIS_ADD", "TO_ORBIS_UPDATE")).select(*K)

print("gold_metadata GENESIS sides on this run :", meta_run.count())
print("funnel TO_ORBIS_ADD + TO_ORBIS_UPDATE    :", pipeline_pub.count())
print("in both                                  :", meta_run.join(pipeline_pub, K).count())
# gold_metadata is the live table, so a later run can have overwritten lastUpdatedOnRunId: small differences are expected for older runs

## 8. Genesis ranges that reach Orbis / are discarded

In [0]:
# Builds the two range lists: to_orbis (reached Orbis) and discarded (dropped, with step and reason).
def export(df):
    return (df.select(
        F.lit(RUN_ID).alias("runId"),
        F.to_json("roadId").alias("roadId"),
        F.col("side_n").alias("side"),
        "countryIso", "streetName", "lr",
        F.col("hn_count").alias("house_numbers"),
        "sourceType", "user",
        "status",
        F.col("delta_op").alias("delta_operation"),
        "violations", "qa_violations"))

to_orbis  = export(ranges.filter(F.col("reaches_orbis")))
discarded = export(ranges.filter(~F.col("reaches_orbis")))
print("to_orbis:", to_orbis.count(), " discarded:", discarded.count())

Quick look / download from the cell output (Databricks lets you download the full result of a `display`).

In [0]:
# Sample of the genesis ranges that reach Orbis.
display(to_orbis.limit(1000))

In [0]:
# Sample of the discarded genesis ranges, with the reason.
display(discarded.limit(1000))

## 9. History of the genesis sides published in the latest run (last 5 runs)

Takes the road sides where **genesis (OTL) won and the delta was `add` or `update` in the latest run**, i.e. genesis ranges that went to Orbis as a change.
For each one it shows what the pipeline did on that road side in each of the last 5 runs, from `adjusted_feature_address_range` (the ranges the delta step compares) and `delta_detected_range`.

Per run, three columns (newest run first):
- `<run_id>_lr`: the LR of each range on that side, prefixed by who won (`OTL` / `ARC`); several ranges are joined with ` | `
- `<run_id>_first_last_hn`: first and last house number of each range, in the same order
- `<run_id>_delta`: `add` / `update` / `delete` / `none` for that side in that run

Empty = the road was not processed in that run. `lr_changes` = how many times the LR value changed between the runs where the side was processed.

In [0]:
# Picks the last 5 runs in the backup (newest first); the newest is the run the sides are selected from.
N_HIST = 5
hist_runs = (spark.table(T_ADJ).select("runId", "backupDate").distinct()
             .withColumn("_d", F.to_date("backupDate", "dd-MM-yyyy"))
             .orderBy(F.col("_d").desc(), F.col("runId").cast("bigint").desc())
             .limit(N_HIST).collect())
HIST_RUN_IDS = [r["runId"] for r in hist_runs]
LAST_RUN = HIST_RUN_IDS[0]
print("last runs (newest first):", [(r["runId"], r["backupDate"]) for r in hist_runs])

In [0]:
# Delta per road side for each of the 5 runs (tags rolled up to the side), and the genesis sides published in the latest run.
HK = ["roadId", "side_n"]

hist_delta = (spark.table(T_DELTA).filter(F.col("runId").isin(HIST_RUN_IDS))
              .withColumnRenamed("id", "roadId")
              .withColumn("side_n", norm_side("side"))
              .withColumn("op", F.lower(F.col("operation").cast("string")))
              .withColumn("scrapped", F.coalesce(F.col("scrapped"), F.lit(False)))
              .groupBy("runId", *HK).agg(
                  F.max((F.col("op") == "add").cast("int")).alias("d_add"),
                  F.max(((F.col("op") == "update") | ((F.col("op") == "delete") & ~F.col("scrapped"))).cast("int")).alias("d_upd"),
                  F.max(((F.col("op") == "delete") & F.col("scrapped")).cast("int")).alias("d_del"))
              .withColumn("delta_op", F.when(F.col("d_add") == 1, "add")
                                       .when(F.col("d_upd") == 1, "update")
                                       .when(F.col("d_del") == 1, "delete")
                                       .otherwise("none"))
              .select("runId", *HK, "delta_op"))

ok_range = (F.coalesce(F.col("violationType"), F.lit("")) != "Error") & (F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0)
adj_hist = (spark.table(T_ADJ).filter(F.col("runId").isin(HIST_RUN_IDS))
            .withColumn("side_n", norm_side("side")).filter(ok_range))

targets = (adj_hist.filter((F.col("runId") == LAST_RUN) & (F.col("preference") == "OTL"))
           .groupBy(*HK).agg(F.first("countryIso").alias("countryIso"), F.first("streetName").alias("streetName"))
           .join(hist_delta.filter((F.col("runId") == LAST_RUN) & F.col("delta_op").isin("add", "update"))
                           .select(*HK, F.col("delta_op").alias("last_run_delta")), HK)
           .cache())
print("genesis sides published (add/update) in run", LAST_RUN, ":", targets.count())
display(targets.groupBy("last_run_delta").count())

In [0]:
# Builds one row per published genesis side with LR, first/last house number and delta for each of the last 5 runs.
def unsigned(c):
    # Orbis ids are unsigned 64-bit, Spark stores them as signed long
    return (F.when(c < 0, c.cast("decimal(20,0)") + F.lit("18446744073709551616").cast("decimal(20,0)"))
             .otherwise(c.cast("decimal(20,0)")).cast("string"))

hn_sorted = F.array_sort(F.transform("lrHouseNumbers", lambda h: F.struct(
    F.split(h["lr"], "-")[0].cast("double").alias("pos"), h["text"].alias("text"))))

per_run = (adj_hist.join(targets.select(*HK), HK)
           .withColumn("hs", hn_sorted)
           .withColumn("fl", F.concat_ws("-", F.element_at("hs", 1)["text"], F.element_at("hs", -1)["text"]))
           .withColumn("start", F.split("lr", "-")[0].cast("double"))
           .groupBy("runId", *HK)
           .agg(F.array_sort(F.collect_list(F.struct("start", "lr", "fl", "preference"))).alias("rs"))
           .withColumn("lr_val", F.concat_ws(" | ", F.transform("rs", lambda r: F.concat_ws(" ", r["preference"], r["lr"]))))
           .withColumn("hn_val", F.concat_ws(" | ", F.transform("rs", lambda r: r["fl"])))
           .select("runId", *HK, "lr_val", "hn_val")
           .join(hist_delta.join(targets.select(*HK), HK), ["runId", *HK], "full"))

pivoted = (per_run.groupBy(*HK).pivot("runId", HIST_RUN_IDS)
           .agg(F.first("lr_val").alias("lr"), F.first("hn_val").alias("first_last_hn"), F.first("delta_op").alias("delta")))

# number of LR changes between consecutive runs where the side was processed (oldest -> newest)
lr_seq = F.filter(F.array(*[F.col(f"{r}_lr") for r in reversed(HIST_RUN_IDS)]), lambda x: x.isNotNull())
lr_changes = F.aggregate(
    lr_seq,
    F.struct(F.lit(None).cast("string").alias("prev"), F.lit(0).alias("n")),
    lambda acc, x: F.struct(x.alias("prev"),
                            (acc["n"] + F.when(acc["prev"].isNotNull() & (acc["prev"] != x), 1).otherwise(0)).alias("n")),
    lambda acc: acc["n"])

run_cols = [c for r in HIST_RUN_IDS for c in (f"{r}_lr", f"{r}_first_last_hn", f"{r}_delta")]
lr_history = (targets.join(pivoted, HK, "left")
              .withColumn("road_id", F.to_json("roadId"))
              .withColumn("unsigned_id", F.concat_ws(":", F.col("roadId.layerId").cast("string"),
                                                     unsigned(F.col("roadId.high")), unsigned(F.col("roadId.low"))))
              .withColumn("lr_changes", lr_changes)
              .select("road_id", "unsigned_id", F.col("side_n").alias("side"), "countryIso", "streetName",
                      F.lit(LAST_RUN).alias("last_run_id"), "last_run_delta", "lr_changes", *run_cols)
              .cache())

display(lr_history.groupBy("last_run_delta", "lr_changes").count().orderBy("last_run_delta", "lr_changes"))
display(lr_history.orderBy(F.desc("lr_changes")).limit(1000))

## 10. Road sides where both ARC and OTL have a range (chosen run)

Uses the run picked in step 2 (`RUN_ID`). Needs cells 1, 3 and 5 to have run.

`ArcOtlRangeMerger` decides per road side: if ARC has a range on that side, ARC wins and the OTL range on that side is dropped.
This section counts how often that conflict happens, using the merge **inputs** after `Merger`'s own filters
(house numbers present, named road, and no genesis violation for OTL; the country scope check is not applied here):
- `feature_address_range` = ARC input, `otl_feature_address_range` = genesis input

It checks left and right separately, shows how each road's left and right side combine, checks that no side has both
after the merge (should be 0), and prints examples with the unsigned id.

In [0]:
# Finds the road sides of RUN_ID where ARC and OTL both have a range before the merge, then counts them for left, right and whole roads.
T_ARC = f"{BACKUP_DB}.feature_address_range"
SK = ["roadId", "side_n"]

def unsigned_id_col(c="roadId"):
    # 6:unsigned_high:unsigned_low, the same format as OrbisIdUtil.convertOrbisIdToString
    def u(x):
        return (F.when(x < 0, x.cast("decimal(20,0)") + F.lit("18446744073709551616").cast("decimal(20,0)"))
                 .otherwise(x.cast("decimal(20,0)")).cast("string"))
    return F.concat_ws(":", F.col(f"{c}.layerId").cast("string"), u(F.col(f"{c}.high")), u(F.col(f"{c}.low")))

def merger_input(t):
    named = F.exists(F.col("tags"), lambda tg: (tg["tagKey"]["key"] == "name") & F.coalesce(F.length(tg["value"]) > 0, F.lit(False)))
    return (spark.table(t).filter(F.col("runId") == RUN_ID)
            .withColumn("side_n", norm_side("side"))
            .filter((F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0) & F.coalesce(named, F.lit(False))))

arc_in = merger_input(T_ARC)
otl_in = merger_input(T_OTL).filter(F.trim(F.coalesce(F.col("violations"), F.lit(""))) == "")

arc_sides = arc_in.groupBy(*SK).agg(F.count("*").alias("arc_ranges"))
otl_sides = otl_in.groupBy(*SK).agg(F.count("*").alias("otl_ranges"))

side_status = (arc_sides.join(otl_sides, SK, "full")
               .withColumn("status", F.when(F.col("arc_ranges").isNotNull() & F.col("otl_ranges").isNotNull(), "BOTH")
                                      .when(F.col("arc_ranges").isNotNull(), "ARC_ONLY")
                                      .otherwise("OTL_ONLY"))
               .cache())

# per side (left / right)
display(side_status.groupBy("side_n").pivot("status", ["BOTH", "ARC_ONLY", "OTL_ONLY"]).count()
        .withColumn("pct_of_otl_sides_lost_to_arc", F.round(100 * F.col("BOTH") / (F.col("BOTH") + F.col("OTL_ONLY")), 2))
        .orderBy("side_n"))

both_sides = side_status.filter(F.col("status") == "BOTH")
print("road sides with both ARC and OTL :", both_sides.count())
print("roads with at least one such side:", both_sides.select("roadId").distinct().count())
print("roads with it on left AND right  :",
      both_sides.groupBy("roadId").agg(F.countDistinct("side_n").alias("n")).filter("n = 2").count())

# how left and right combine on each road (NONE = that side has no range in either input)
road_lr = (side_status.groupBy("roadId").pivot("side_n", ["left", "right"]).agg(F.first("status"))
           .fillna("NONE", subset=["left", "right"]))
display(road_lr.groupBy("left", "right").count().orderBy(F.desc("count")))

In [0]:
# Sanity check on the merge output: no road side should have both an ARC and an OTL range after ArcOtlRangeMerger (expected 0 for both tables).
for t in [T_MERGED, T_ADJ]:
    n = (spark.table(t).filter(F.col("runId") == RUN_ID).withColumn("side_n", norm_side("side"))
         .groupBy(*SK).agg(F.countDistinct("preference").alias("n_pref")).filter("n_pref > 1").count())
    print(f"{t}: sides with both ARC and OTL after merge = {n}")

In [0]:
# Examples: road sides where ARC and OTL both had a range, with the unsigned id and each source's LR and first-last house number.
def ranges_text(df, label):
    hs = F.array_sort(F.transform("lrHouseNumbers", lambda h: F.struct(
        F.split(h["lr"], "-")[0].cast("double").alias("pos"), h["text"].alias("text"))))
    return (df.withColumn("hs", hs)
              .withColumn("start", F.split("lr", "-")[0].cast("double"))
              .withColumn("txt", F.concat(F.col("lr"), F.lit(" ["),
                                          F.concat_ws("-", F.element_at("hs", 1)["text"], F.element_at("hs", -1)["text"]),
                                          F.lit("]")))
              .groupBy(*SK).agg(F.first("countryIso").alias(f"{label}_country"),
                                F.first("streetName").alias(f"{label}_street"),
                                F.array_sort(F.collect_list(F.struct("start", "txt"))).alias("rs"))
              .withColumn(f"{label}_lr_hn", F.concat_ws(" | ", F.transform("rs", lambda r: r["txt"])))
              .drop("rs"))

examples = (both_sides.limit(50)
            .join(ranges_text(arc_in, "arc"), SK)
            .join(ranges_text(otl_in, "otl"), SK)
            .select(unsigned_id_col().alias("unsigned_id"), F.col("side_n").alias("side"),
                    F.coalesce("arc_country", "otl_country").alias("countryIso"),
                    F.col("arc_street"), F.col("otl_street"),
                    "arc_ranges", "otl_ranges", "arc_lr_hn", "otl_lr_hn"))
display(examples)
for r in examples.limit(10).collect():
    print(f"{r['unsigned_id']}  {r['side']:5}  {r['countryIso']}  ARC: {r['arc_lr_hn']}  ||  OTL: {r['otl_lr_hn']}")

## 11. Which genesis ranges are worth a passar call? (chosen run)

Uses the run picked in step 2 (`RUN_ID`). Needs cells 1, 3 and 5 to have run.

ARC builds its ranges from APTs, and ARC wins any road side where it has a range. So a genesis (OTL) range is only useful on a
side **without APTs**. This section measures that for one run:

1. genesis ranges per side (left / right)
2. are their roads in `eligible_roadelements_incremental`?
3. of the eligible ones, how many reach Orbis after the merge (OTL won, no QA error, delta `add` / `update` / `none`)
4. APTs per road side, from `master_data` (`addressPoints[].side`): APT on both sides, left only, right only, none
5. per genesis side: does it have APTs on that side, and did it reach Orbis or lose to ARC?
6. what that means for passar calls: roads where genesis could be skipped vs roads that still need it

In [0]:
# 1. Genesis ranges, road sides and roads in otl_feature_address_range for RUN_ID, per side.
T_ELIG = f"{BACKUP_DB}.eligible_roadelements_incremental"
T_MASTER = f"{BACKUP_DB}.master_data"
SK = ["roadId", "side_n"]

otl_run = spark.table(T_OTL).filter(F.col("runId") == RUN_ID).withColumn("side_n", norm_side("side")).cache()

display(otl_run.groupBy("side_n").agg(F.count("*").alias("otl_ranges"),
                                      F.countDistinct("roadId").alias("roads"))
        .union(otl_run.agg(F.lit("TOTAL"), F.count("*"), F.countDistinct("roadId")))
        .orderBy("side_n"))

In [0]:
# 2. Is each genesis road in eligible_roadelements_incremental for the same run?
elig_roads = (spark.table(T_ELIG).filter(F.col("runId") == RUN_ID)
              .select("roadId").distinct().withColumn("eligible", F.lit(True)))

otl_elig = otl_run.join(elig_roads, "roadId", "left").fillna(False, subset=["eligible"]).cache()

display(otl_elig.groupBy("eligible").agg(F.count("*").alias("otl_ranges"),
                                         F.countDistinct(*SK).alias("road_sides"),
                                         F.countDistinct("roadId").alias("roads")))

In [0]:
# 3. Of the eligible genesis road sides, how many reach Orbis after the merge, and what happened to the rest.
not_err = F.coalesce(F.col("violationType"), F.lit("")) != "Error"
has_hn = F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0

def run_sides(t, cond):
    return (spark.table(t).filter(F.col("runId") == RUN_ID).withColumn("side_n", norm_side("side"))
            .filter(cond).select(*SK).distinct())

arc_won = run_sides(T_MERGED, F.col("preference") == "ARC").withColumn("arc_won", F.lit(True))
otl_won = run_sides(T_MERGED, F.col("preference") == "OTL").withColumn("otl_won", F.lit(True))
otl_final = run_sides(T_ADJ, (F.col("preference") == "OTL") & not_err & has_hn).withColumn("otl_final", F.lit(True))
delta_side = (spark.table(T_DELTA).filter(F.col("runId") == RUN_ID).withColumnRenamed("id", "roadId")
              .withColumn("side_n", norm_side("side"))
              .withColumn("op", F.lower(F.col("operation").cast("string")))
              .groupBy(*SK).agg(F.max((F.col("op").isin("add", "update", "none")).cast("int")).alias("in_delta")))

otl_side_outcome = (otl_elig.select(*SK, "eligible").distinct()
    .join(arc_won, SK, "left").join(otl_won, SK, "left").join(otl_final, SK, "left").join(delta_side, SK, "left")
    .fillna(False, subset=["arc_won", "otl_won", "otl_final"]).fillna(0, subset=["in_delta"])
    .withColumn("outcome",
        F.when(~F.col("eligible"), "NOT_ELIGIBLE_ROAD")
         .when(F.col("arc_won"), "ARC_WINS")
         .when(~F.col("otl_won"), "DROPPED_BEFORE_MERGE")
         .when(~F.col("otl_final"), "DROPPED_QA_OVERLAP_ADJUST")
         .when(F.col("in_delta") == 1, "TO_ORBIS")
         .otherwise("NO_DELTA"))
    .cache())

display(otl_side_outcome.groupBy("eligible", "outcome").agg(F.count("*").alias("road_sides"),
                                                           F.countDistinct("roadId").alias("roads"))
        .withColumn("pct_of_sides", F.round(100 * F.col("road_sides") / F.sum("road_sides").over(Window.partitionBy()), 2))
        .orderBy(F.desc("eligible"), F.desc("road_sides")))

In [0]:
# 4. APTs per road side for the genesis roads, from master_data.addressPoints[].side (a road missing from master_data has no APTs).
apt_sides = (spark.table(T_MASTER).filter(F.col("runId") == RUN_ID)
             .select("roadId", F.explode_outer("addressPoints").alias("ap"))
             .filter(F.col("ap").isNotNull())
             .withColumn("apt_side", F.lower(F.regexp_extract(F.col("ap.side").cast("string"), "(?i)(left|right|both)", 1)))
             .groupBy("roadId").agg(
                 F.count("*").alias("apts"),
                 F.max(F.col("apt_side").isin("left", "both").cast("int")).alias("apt_left"),
                 F.max(F.col("apt_side").isin("right", "both").cast("int")).alias("apt_right")))

otl_roads_apt = (otl_run.select("roadId").distinct()
                 .join(apt_sides, "roadId", "left").fillna(0, subset=["apts", "apt_left", "apt_right"])
                 .withColumn("apt_on",
                     F.when((F.col("apt_left") == 1) & (F.col("apt_right") == 1), "BOTH_SIDES")
                      .when(F.col("apt_left") == 1, "LEFT_ONLY")
                      .when(F.col("apt_right") == 1, "RIGHT_ONLY")
                      .otherwise("NO_APT"))
                 .cache())

display(otl_roads_apt.groupBy("apt_on").agg(F.count("*").alias("genesis_roads"), F.sum("apts").alias("apts"))
        .withColumn("pct_of_genesis_roads", F.round(100 * F.col("genesis_roads") / F.sum("genesis_roads").over(Window.partitionBy()), 2))
        .orderBy(F.desc("genesis_roads")))

In [0]:
# 5. Per genesis road side: are there APTs on that side, and did the genesis range reach Orbis or lose to ARC?
side_apt = (otl_side_outcome.join(otl_roads_apt.select("roadId", "apt_left", "apt_right"), "roadId", "left")
            .withColumn("apt_on_this_side",
                F.when(F.col("side_n") == "left", F.col("apt_left") == 1)
                 .when(F.col("side_n") == "right", F.col("apt_right") == 1)
                 .otherwise((F.col("apt_left") == 1) | (F.col("apt_right") == 1)))
            .fillna(False, subset=["apt_on_this_side"]))

display(side_apt.groupBy("side_n", "apt_on_this_side").pivot("outcome").count().orderBy("side_n", "apt_on_this_side"))

display(side_apt.groupBy("apt_on_this_side").agg(
            F.count("*").alias("genesis_sides"),
            F.sum((F.col("outcome") == "TO_ORBIS").cast("int")).alias("to_orbis"),
            F.sum((F.col("outcome") == "ARC_WINS").cast("int")).alias("arc_wins"))
        .withColumn("pct_to_orbis", F.round(100 * F.col("to_orbis") / F.col("genesis_sides"), 2)))

In [0]:
# 6. Passar view, per road: if genesis were only called for roads with at least one side without APTs,
#    how many calls would be skipped, and how many genesis sides that reached Orbis would be lost.
road_view = (side_apt.groupBy("roadId").agg(
                 F.max((F.col("outcome") == "TO_ORBIS").cast("int")).alias("any_to_orbis"),
                 F.sum((F.col("outcome") == "TO_ORBIS").cast("int")).alias("sides_to_orbis"))
             .join(otl_roads_apt.select("roadId", "apt_on"), "roadId"))

summary = (road_view.groupBy("apt_on").agg(
               F.count("*").alias("genesis_roads"),
               F.sum("any_to_orbis").alias("roads_with_genesis_in_orbis"),
               F.sum("sides_to_orbis").alias("genesis_sides_in_orbis"))
           .withColumn("roads_where_call_was_wasted", F.col("genesis_roads") - F.col("roads_with_genesis_in_orbis"))
           .orderBy("apt_on"))
display(summary)

tot = road_view.count()
skip = road_view.filter(F.col("apt_on") == "BOTH_SIDES")
n_skip = skip.count()
lost = skip.agg(F.sum("sides_to_orbis")).first()[0] or 0
all_in_orbis = road_view.agg(F.sum("sides_to_orbis")).first()[0] or 0
print(f"genesis roads in run {RUN_ID}: {tot}")
print(f"skip roads with APTs on BOTH sides -> {n_skip} calls avoided ({100 * n_skip / max(tot, 1):.1f}%), "
      f"losing {lost} of {all_in_orbis} genesis sides that reached Orbis ({100 * lost / max(all_in_orbis, 1):.1f}%)")
print("roads with APTs on one side only still need genesis for the other side; ask passar for that side only if the API allows it")

In [0]:
# Examples: genesis sides that reached Orbis although that side has APTs (ARC had no range there), with the unsigned id.
def uid(c="roadId"):
    def u(x):
        return (F.when(x < 0, x.cast("decimal(20,0)") + F.lit("18446744073709551616").cast("decimal(20,0)"))
                 .otherwise(x.cast("decimal(20,0)")).cast("string"))
    return F.concat_ws(":", F.col(f"{c}.layerId").cast("string"), u(F.col(f"{c}.high")), u(F.col(f"{c}.low")))

display(side_apt.filter((F.col("outcome") == "TO_ORBIS") & F.col("apt_on_this_side"))
        .join(otl_roads_apt.select("roadId", "apts", "apt_on"), "roadId")
        .select(uid().alias("unsigned_id"), F.col("side_n").alias("side"), "apt_on", "apts", "outcome")
        .limit(50))

## 12. Road elements where ARC and OTL both have a range on the same side (chosen run)

Uses the run picked in step 2 (`RUN_ID`). Needs cells 1, 3 and 5 to have run.

A **road element** (`roadId`) is counted when, on the **same side** (left or right), it has at least one ARC range
(`feature_address_range`) **and** at least one OTL / genesis range (`otl_feature_address_range`) in this run.
`ArcOtlRangeMerger` only looks at `left` / `right`, so other side values are ignored.

Two views:
- `raw`: every range in the backup tables for the run, no filter
- `merger_input`: after `Merger`'s input filters (house numbers present, named road, and no genesis violation for OTL; country scope not applied). This is what `ArcOtlRangeMerger` actually sees, the same input as section 10.

Cell 1 counts them, cell 2 counts them per country, cell 3 lists each road element with ARC and OTL LR + first-last house number per side.

In [0]:
# 12.1 Counts road elements (distinct roadId) and road sides where ARC and OTL both have a range on the same side, for both views.
T_ARC = f"{BACKUP_DB}.feature_address_range"
SK = ["roadId", "side_n"]

def run_ranges(t):
    return (spark.table(t).filter(F.col("runId") == RUN_ID)
            .withColumn("side_n", norm_side("side"))
            .filter(F.col("side_n").isin("left", "right")))

def merger_filter(df, is_otl):
    named = F.exists(F.col("tags"), lambda tg: (tg["tagKey"]["key"] == "name") & F.coalesce(F.length(tg["value"]) > 0, F.lit(False)))
    cond = (F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0) & F.coalesce(named, F.lit(False))
    if is_otl:
        cond = cond & (F.trim(F.coalesce(F.col("violations"), F.lit(""))) == "")
    return df.filter(cond)

arc_raw, otl_raw = run_ranges(T_ARC), run_ranges(T_OTL)
same_side_views = {
    "raw": (arc_raw, otl_raw),
    "merger_input": (merger_filter(arc_raw, False), merger_filter(otl_raw, True)),
}

def same_side(arc, otl):
    # inner join on (roadId, side) = ARC and OTL on the same road element and the same side
    a = arc.groupBy(*SK).agg(F.count("*").alias("arc_ranges"))
    o = otl.groupBy(*SK).agg(F.count("*").alias("otl_ranges"))
    return a.join(o, SK)

same_side_sides = {v: same_side(a, o).cache() for v, (a, o) in same_side_views.items()}

stats = None
for v, df in same_side_sides.items():
    _, otl_v = same_side_views[v]
    both_lr = (df.groupBy("roadId").agg(F.countDistinct("side_n").alias("n"))
                 .agg(F.sum((F.col("n") == 2).cast("int")).alias("road_elements_left_and_right")))
    s = (df.agg(F.countDistinct("roadId").alias("road_elements_same_side"),
                F.count("*").alias("road_sides_same_side"),
                F.sum((F.col("side_n") == "left").cast("int")).alias("left_sides"),
                F.sum((F.col("side_n") == "right").cast("int")).alias("right_sides"),
                F.sum("arc_ranges").alias("arc_ranges_on_those_sides"),
                F.sum("otl_ranges").alias("otl_ranges_on_those_sides"))
           .crossJoin(both_lr)
           .crossJoin(otl_v.agg(F.countDistinct("roadId").alias("otl_road_elements")))
           .withColumn("pct_of_otl_road_elements",
                       F.round(100 * F.col("road_elements_same_side") / F.col("otl_road_elements"), 2))
           .select(F.lit(v).alias("view"), "*"))
    stats = s if stats is None else stats.unionByName(s)

display(stats.fillna(0))

In [0]:
# 12.2 Same as 12.1, per country (merger_input view): road elements and road sides where ARC and OTL are on the same side.
VIEW = "merger_input"   # or "raw"

arc_v, otl_v = same_side_views[VIEW]
road_country = (arc_v.select("roadId", "countryIso").unionByName(otl_v.select("roadId", "countryIso"))
                .groupBy("roadId").agg(F.first("countryIso", ignorenulls=True).alias("countryIso")))

display(same_side_sides[VIEW].join(road_country, "roadId", "left")
        .groupBy("countryIso")
        .agg(F.countDistinct("roadId").alias("road_elements"),
             F.count("*").alias("road_sides"),
             F.sum((F.col("side_n") == "left").cast("int")).alias("left_sides"),
             F.sum((F.col("side_n") == "right").cast("int")).alias("right_sides"))
        .orderBy(F.desc("road_elements")))

In [0]:
# 12.3 Lists every road element (VIEW from 12.2) where ARC and OTL are on the same side: one row per road element,
#      with the ARC and OTL ranges (LR [first-last house number]) on the left and right side.
def _uid(c="roadId"):
    # 6:unsigned_high:unsigned_low, same format as OrbisIdUtil.convertOrbisIdToString
    def u(x):
        return (F.when(x < 0, x.cast("decimal(20,0)") + F.lit("18446744073709551616").cast("decimal(20,0)"))
                 .otherwise(x.cast("decimal(20,0)")).cast("string"))
    return F.concat_ws(":", F.col(f"{c}.layerId").cast("string"), u(F.col(f"{c}.high")), u(F.col(f"{c}.low")))

same = same_side_sides[VIEW]

def side_text(df, label):
    hs = F.array_sort(F.transform("lrHouseNumbers", lambda h: F.struct(
        F.split(h["lr"], "-")[0].cast("double").alias("pos"), h["text"].alias("text"))))
    return (df.join(same.select(*SK), SK)
              .withColumn("hs", hs)
              .withColumn("start", F.split("lr", "-")[0].cast("double"))
              .withColumn("txt", F.concat(F.col("lr"), F.lit(" ["),
                                          F.concat_ws("-", F.element_at("hs", 1)["text"], F.element_at("hs", -1)["text"]),
                                          F.lit("]")))
              .groupBy(*SK).agg(F.first("countryIso", ignorenulls=True).alias(f"{label}_country"),
                                F.first("streetName", ignorenulls=True).alias(f"{label}_street"),
                                F.array_sort(F.collect_list(F.struct("start", "txt"))).alias("rs"))
              .withColumn(f"{label}_lr_hn", F.concat_ws(" | ", F.transform("rs", lambda r: r["txt"])))
              .drop("rs"))

side_detail = same.join(side_text(arc_v, "arc"), SK, "left").join(side_text(otl_v, "otl"), SK, "left")

per_side = (side_detail.groupBy("roadId").pivot("side_n", ["left", "right"])
            .agg(F.first("arc_ranges").alias("arc_ranges"), F.first("otl_ranges").alias("otl_ranges"),
                 F.first("arc_lr_hn").alias("arc_lr_hn"), F.first("otl_lr_hn").alias("otl_lr_hn")))

road_info = side_detail.groupBy("roadId").agg(
    F.first(F.coalesce("arc_country", "otl_country"), ignorenulls=True).alias("countryIso"),
    F.first("arc_street", ignorenulls=True).alias("arc_street"),
    F.first("otl_street", ignorenulls=True).alias("otl_street"),
    F.concat_ws("+", F.array_sort(F.collect_set("side_n"))).alias("same_sides"))

same_side_roads = (road_info.join(per_side, "roadId")
                   .select(_uid().alias("unsigned_id"), F.to_json("roadId").alias("road_id"),
                           "countryIso", "arc_street", "otl_street", "same_sides",
                           "left_arc_ranges", "left_otl_ranges", "left_arc_lr_hn", "left_otl_lr_hn",
                           "right_arc_ranges", "right_otl_ranges", "right_arc_lr_hn", "right_otl_lr_hn")
                   .cache())

print(f"[{VIEW}] road elements with ARC and OTL on the same side in run {RUN_ID}: {same_side_roads.count()}")
display(same_side_roads.groupBy("same_sides").count().orderBy(F.desc("count")))
display(same_side_roads.orderBy("countryIso", "unsigned_id"))

### 12.4 Same count on the final result (after the merge and the ARC / OTL preference)

Uses the pipeline output instead of the merge inputs: every range carries `preference` = `ARC` or `OTL`.
- `merged`: `feature_address_range_merged`, straight after `ArcOtlRangeMerger`
- `final`: `adjusted_feature_address_range` without `Error` ranges and with house numbers, i.e. the ranges the delta step publishes

`same_road_same_side` = road elements with an ARC **and** an OTL range on the same side. `ArcOtlRangeMerger` gives each side to one source, so this should be **0**; anything else is listed below.
`same_road_other_side` = road elements with ARC on one side and OTL on the other (allowed, shown for context).

In [0]:
# 12.4 Road elements with both an ARC and an OTL range on the same side in the final result (expected 0), and with ARC / OTL on different sides.
not_err = F.coalesce(F.col("violationType"), F.lit("")) != "Error"
has_hn = F.coalesce(F.size("lrHouseNumbers"), F.lit(0)) > 0

def run_out(t, cond=F.lit(True)):
    return (spark.table(t).filter(F.col("runId") == RUN_ID)
            .withColumn("side_n", norm_side("side"))
            .filter(F.col("side_n").isin("left", "right") & F.col("preference").isin("ARC", "OTL") & cond))

final_views = {"merged": run_out(T_MERGED), "final": run_out(T_ADJ, not_err & has_hn)}

final_sides = {}
fstats = None
for v, df in final_views.items():
    side_pref = (df.groupBy(*SK).agg(F.count(F.when(F.col("preference") == "ARC", 1)).alias("arc_ranges"),
                                     F.count(F.when(F.col("preference") == "OTL", 1)).alias("otl_ranges"))
                 .cache())
    final_sides[v] = side_pref
    road = side_pref.groupBy("roadId").agg(
        F.max(((F.col("arc_ranges") > 0) & (F.col("otl_ranges") > 0)).cast("int")).alias("same_side"),
        F.max((F.col("arc_ranges") > 0).cast("int")).alias("has_arc"),
        F.max((F.col("otl_ranges") > 0).cast("int")).alias("has_otl"))
    s = (road.agg(F.count("*").alias("road_elements"),
                  F.sum("has_arc").alias("road_elements_with_arc"),
                  F.sum("has_otl").alias("road_elements_with_otl"),
                  F.sum("same_side").alias("same_road_same_side"),
                  F.sum(((F.col("has_arc") == 1) & (F.col("has_otl") == 1) & (F.col("same_side") == 0)).cast("int")).alias("same_road_other_side"))
           .crossJoin(side_pref.agg(F.sum(((F.col("arc_ranges") > 0) & (F.col("otl_ranges") > 0)).cast("int")).alias("road_sides_same_side")))
           .select(F.lit(v).alias("view"), "*"))
    fstats = s if fstats is None else fstats.unionByName(s)

display(fstats.fillna(0))

# list the road elements where the same side still has both ARC and OTL in the final result (should be empty)
conflict = final_sides["final"].filter((F.col("arc_ranges") > 0) & (F.col("otl_ranges") > 0))
final_df = final_views["final"]
conflict_list = (final_df.join(conflict.select(*SK), SK)
                 .groupBy(*SK).agg(F.first("countryIso", ignorenulls=True).alias("countryIso"),
                                   F.first("streetName", ignorenulls=True).alias("streetName"),
                                   F.concat_ws(" | ", F.array_sort(F.collect_list(F.concat_ws(" ", "preference", "lr")))).alias("pref_lr"))
                 .join(conflict, SK)
                 .select(_uid().alias("unsigned_id"), F.to_json("roadId").alias("road_id"), F.col("side_n").alias("side"),
                         "countryIso", "streetName", "arc_ranges", "otl_ranges", "pref_lr"))
print(f"[final] road elements with ARC and OTL on the same side in run {RUN_ID}:", conflict_list.select("road_id").distinct().count())
display(conflict_list.orderBy("countryIso", "unsigned_id"))

## 13. Roads with APTs but no house number on any APT, and no address range (chosen run)

Uses the run picked in step 2 (`RUN_ID`). Needs cells 1, 3 and 5 to have run.

Source: `master_data` (one row per eligible road, `addressPoints[]` = the APTs hooked to it). The pipeline stores the APT's house number in
`addressPoints[].value` (`IndexPlacesDataForRoadV2.getHouseNumber`): `addr:housenumber:<lang>`, or `addr:streetnumber:<lang>` / the conscription number in
the street-number countries. An empty `value` = the APT has **no `addr:housenumber`**.

A road is counted when:
1. it has at least one APT in `master_data`,
2. **none** of its APTs has a house number (`apts_without_hn = apts`), and
3. it has **no address range** in this run on either side: not in `feature_address_range` (ARC), `otl_feature_address_range` (genesis)
   or `adjusted_feature_address_range` (final ranges).

`roads_some_apts_without_hn_no_range` is shown for context: at least one APT without a house number (not all), and still no range.

Cell 13.1 builds the road list, 13.2 counts per country, 13.3 prints samples per country.

In [0]:
# 13.1 Per road in master_data for RUN_ID: APT count, APTs without house number, and whether the road has any address range in this run.
T_MASTER = f"{BACKUP_DB}.master_data"
T_ARC = f"{BACKUP_DB}.feature_address_range"

def _uid13(c="roadId"):
    # 6:unsigned_high:unsigned_low, same format as OrbisIdUtil.convertOrbisIdToString
    def u(x):
        return (F.when(x < 0, x.cast("decimal(20,0)") + F.lit("18446744073709551616").cast("decimal(20,0)"))
                 .otherwise(x.cast("decimal(20,0)")).cast("string"))
    return F.concat_ws(":", F.col(f"{c}.layerId").cast("string"), u(F.col(f"{c}.high")), u(F.col(f"{c}.low")))

no_hn = F.trim(F.coalesce(F.col("ap.value"), F.lit(""))) == ""

road_apts = (spark.table(T_MASTER).filter(F.col("runId") == RUN_ID)
             .select("roadId", F.col("Country_ISO").alias("countryIso"), "streetName",
                     F.explode("addressPoints").alias("ap"))           # roads without APTs drop out here
             .withColumn("apt_side", F.lower(F.regexp_extract(F.col("ap.side").cast("string"), "(?i)(left|right|both)", 1)))
             .groupBy("roadId")
             .agg(F.first("countryIso", ignorenulls=True).alias("countryIso"),
                  F.first("streetName", ignorenulls=True).alias("streetName"),
                  F.count("*").alias("apts"),
                  F.sum(no_hn.cast("int")).alias("apts_without_hn"),
                  F.concat_ws(",", F.array_sort(F.collect_set("apt_side"))).alias("apt_sides"),
                  F.slice(F.collect_list(F.to_json("ap.aptId")), 1, 3).alias("sample_apt_ids")))

def roads_with_range(t, cond=F.lit(True)):
    return spark.table(t).filter((F.col("runId") == RUN_ID) & cond).select("roadId").distinct()

has_range = (roads_with_range(T_ARC)
             .union(roads_with_range(T_OTL))
             .union(roads_with_range(T_ADJ))
             .distinct().withColumn("has_range", F.lit(True)))

apt_roads = (road_apts.join(has_range, "roadId", "left")
             .fillna(False, subset=["has_range"])
             .withColumn("all_apts_without_hn", F.col("apts_without_hn") == F.col("apts"))
             .withColumn("some_apts_without_hn", (F.col("apts_without_hn") > 0) & (F.col("apts_without_hn") < F.col("apts")))
             .cache())

# the roads this section is about
no_hn_no_range = apt_roads.filter(F.col("all_apts_without_hn") & ~F.col("has_range")).cache()

print("roads with APTs in master_data                       :", apt_roads.count())
print("  of which all APTs without house number             :", apt_roads.filter("all_apts_without_hn").count())
print("  of which all APTs without house number AND no range:", no_hn_no_range.count())

In [0]:
# 13.2 Country-wise count of roads with APTs where no APT has a house number and the road has no address range.
country_counts = (apt_roads.groupBy("countryIso").agg(
        F.count("*").alias("roads_with_apts"),
        F.sum(F.col("all_apts_without_hn").cast("int")).alias("roads_all_apts_without_hn"),
        F.sum((F.col("all_apts_without_hn") & ~F.col("has_range")).cast("int")).alias("roads_all_apts_without_hn_no_range"),
        F.sum(F.when(F.col("all_apts_without_hn") & ~F.col("has_range"), F.col("apts")).otherwise(0)).alias("apts_on_those_roads"),
        F.sum((F.col("some_apts_without_hn") & ~F.col("has_range")).cast("int")).alias("roads_some_apts_without_hn_no_range"))
    .withColumn("pct_of_roads_with_apts",
                F.round(100 * F.col("roads_all_apts_without_hn_no_range") / F.col("roads_with_apts"), 2))
    .filter((F.col("roads_all_apts_without_hn_no_range") > 0) | (F.col("roads_some_apts_without_hn_no_range") > 0))
    .orderBy(F.desc("roads_all_apts_without_hn_no_range")))
display(country_counts)

tot = no_hn_no_range.agg(F.count("*").alias("roads"), F.sum("apts").alias("apts")).first()
print(f"run {RUN_ID}: {tot['roads']} roads, {tot['apts'] or 0} APTs, no house number on any APT and no address range")

In [0]:
# 13.3 Samples per country: up to N_SAMPLES roads per country (most APTs first), with the unsigned id, street, APT sides and a few APT ids.
N_SAMPLES = 5

w = Window.partitionBy("countryIso").orderBy(F.desc("apts"), F.col("roadId.low"))
samples = (no_hn_no_range.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") <= N_SAMPLES)
           .select("countryIso", _uid13().alias("unsigned_id"), F.to_json("roadId").alias("road_id"),
                   "streetName", "apts", "apts_without_hn", "apt_sides",
                   F.concat_ws(" ; ", "sample_apt_ids").alias("sample_apt_ids"))
           .orderBy("countryIso", F.desc("apts"))
           .cache())
display(samples)

current = None
for r in samples.collect():
    if r["countryIso"] != current:
        current = r["countryIso"]
        print(f"\n=== {current} ===")
    print(f"  {r['unsigned_id']}  {(r['streetName'] or ''):<35.35}  apts={r['apts']:<4} sides={r['apt_sides']:<12} apt_ids={r['sample_apt_ids']}")